# Setup, correctness checks, and GPU profiling

Select the RTX PRO 6000 GPU in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated"
DATA = WORKSPACE / "data" / "fineweb_edu"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu"
print("Experiment outputs:", RUNS)


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        try:
            torch_version = version("torch")
        except PackageNotFoundError as error:
            raise RuntimeError("Select a Colab runtime with PyTorch installed and rerun setup.") from error

        # Keep the runtime's PyTorch build, including its CUDA version suffix.
        extras = "data,eval" if benchmark else "data"
        print(f"Using {sys.executable} with PyTorch {torch_version}", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"torch=={torch_version}",
                "-e",
                f"{repo}[{extras}]",
            ],
            check=True,
        )
    return repo

REPO = setup_colab(WORKSPACE, RUNS, benchmark=False)

def run(*arguments):
    subprocess.run([sys.executable, "-m", "base_model_research.cli", *map(str, arguments)],
                   cwd=REPO, check=True)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
run("environment", "--output", RUNS / "environment.json")
run("check", "--device", "cpu", "--output", RUNS / "checks.json")
run("trials")


Run these GPU diagnostics before full sweeps. Inspect activation magnitudes across widths for collapse or growth; the new gated architecture has no guaranteed transfer result.


In [ ]:
for family in ("standard", "modified"):
    run("coordinate-check", "--model", f"configs/models/{family}_proxy.yaml",
        "--output", RUNS / f"{family}_coordinates.json")


This cell profiles all four models using synthetic data. If a model runs out of GPU memory, reduce the microbatch size and rerun the cell. Training uses each model's saved settings.


In [ ]:
MICROBATCH_SIZE = 4  # Must divide the global batch size of 128.
for model in ("standard_proxy", "modified_proxy", "standard_main", "modified_main"):
    print(f"Profiling {model}…", flush=True)
    run("profile", "--model", f"configs/models/{model}.yaml",
        "--microbatch-size", MICROBATCH_SIZE, "--output", RUNS / f"{model}_profile.json")
